# OpenAI API Codex-style Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kingwabg/supergemma4-colab-runner/blob/main/notebooks/OpenAI_API_Codex_Style_Colab.ipynb)

이 노트북은 Codex/OpenAI 모델을 Colab GPU에 직접 올리는 방식이 아닙니다. Colab을 작업장으로 쓰고, OpenAI API로 모델을 호출하는 방식입니다.

GPU는 필요 없습니다. `Runtime type`은 `Python 3`, 하드웨어 가속기는 `CPU`로 둬도 됩니다.

주의:

- API 사용량에 따라 OpenAI API 비용이 발생할 수 있습니다.
- API 키를 노트북에 저장하지 마세요. 아래 셀은 `getpass()`로 실행 중에만 입력받습니다.
- 비밀키, 주민번호, 계정 비밀번호 같은 민감정보를 프롬프트에 넣지 마세요.


## 1. 설치

OpenAI Python SDK를 설치합니다.


In [ ]:
%pip install -q -U openai


## 2. API 키 입력

OpenAI API 키는 https://platform.openai.com/api-keys 에서 만들 수 있습니다.

아래 셀을 실행하면 입력창이 뜹니다. 키는 화면에 표시되지 않습니다.


In [ ]:
import os
from getpass import getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('OpenAI API key: ')

print('API 키가 런타임 환경변수에 설정되었습니다.')


## 3. 연결 테스트

짧은 요청으로 API 연결을 확인합니다. 모델 접근 권한이 없으면 `MODEL` 값을 바꿔서 다시 실행하세요.


In [ ]:
from openai import OpenAI

client = OpenAI()

# 계정 권한에 따라 사용 가능한 모델이 다를 수 있습니다.
# 공식 Responses API 예시는 gpt-4.1을 사용합니다.
# 최신 GPT-5 계열 접근 권한이 있으면 예: MODEL = 'gpt-5.6' 으로 바꿔보세요.
MODEL = 'gpt-4.1'

response = client.responses.create(
    model=MODEL,
    instructions='한국어로 짧고 정확하게 답하세요.',
    input='Colab에서 OpenAI API 연결 테스트 중입니다. 한 문장으로 응답하세요.',
    max_output_tokens=120,
)

print(response.output_text)


## 4. Codex 스타일 코딩 도우미

아래 셀은 코딩 작업용 시스템 지시문을 붙인 간단한 채팅 루프입니다.

- 질문 입력: `나:` 입력창에 작성 후 Enter
- 종료: `/exit` 또는 `/quit`
- 대화는 이 Colab 런타임 메모리에만 보관됩니다.


In [ ]:
CODING_INSTRUCTIONS = '''
당신은 실용적인 코딩 도우미입니다.
한국어로 답하고, 먼저 결론을 말한 뒤 필요한 코드와 실행 순서를 제시하세요.
불확실한 부분은 추측이라고 표시하세요.
보안상 위험한 명령이나 비밀키 노출은 경고하세요.
'''

conversation_input = []

print('준비 완료. 종료하려면 /exit 또는 /quit 입력')

while True:
    user_text = input('\n나: ').strip()
    if user_text.lower() in {'/exit', '/quit'}:
        print('종료합니다.')
        break
    if not user_text:
        continue

    conversation_input.append({'role': 'user', 'content': user_text})

    response = client.responses.create(
        model=MODEL,
        instructions=CODING_INSTRUCTIONS,
        input=conversation_input,
        max_output_tokens=1200,
    )

    answer = response.output_text.strip()
    print('\nAI:\n' + answer)
    conversation_input.append({'role': 'assistant', 'content': answer})


## 5. Colab 코드 실행 보조

모델이 코드를 제안하면, 아래 셀을 새로 추가해서 직접 실행하세요. API 모델이 Colab을 자동 제어하는 것은 아니고, 이 노트북에서는 사용자가 실행 여부를 결정합니다.

더 강한 자동화가 필요하면 별도 에이전트/MCP 구성이 필요합니다.
